# PlantVision 2.0 — Houseplant disease pipeline

**Stage 1:** YOLOv8n single-class `plant_leaf` detector — shipped pre-trained in `server/models/detector.{pt,onnx}`. §3 re-trains only when a YOLO dataset is present under `data_v2/`.

**Stage 2:** EfficientNetV2-S teacher → distilled MobileNetV3-Large student. v2.0 ships **7 classes**: `bacterial_leaf_spot`, `gray_mold_botrytis`, `healthy`, `leaf_spot_fungal`, `not_a_plant`, `powdery_mildew`, `viral_disease`. Four physiological conditions (`edema`, `leaf_scorch_underwatering`, `nutrient_deficiency`, `sunburn_or_scorch`) are deferred to v2.1: iNat indexes pathogens, not symptoms, so there is no public taxon source for them. Keeping them in the softmax with zero training data would only produce random argmax predictions — better to under-promise and add them with real recall later.

**Data — three quality public sources, no manual upload:** iNaturalist research-grade leaves, COCO val2017 → `not_a_plant`, PlantVillage HF mirror mapped onto our taxonomy. The Roboflow path was removed; PlantVillage is split deterministically by stem-hash so its `val_studio` slice cannot leak into train. `val_realworld` (the ship/no-ship gate) is a 15 % group-aware slice of train.

**Primary metric:** disease macro-F1 on `val_realworld`. Target 0.70–0.78, no class with recall=0, healthy-bias < 1.5.

**Time budget on 1× A100-40:** ≈10 min PV+iNat fetch, ≈5 min DINOv2 probe, ≈35 min teacher, ≈10 min KD, ≈5 min INT8 + temp + conformal.

In [ ]:
# §1 — Setup: deps, reproducibility, schema, layout, eval helpers
import sys, subprocess, random, json, hashlib, pathlib, re, os, shutil, time, zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Optional

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch>=2.2', 'torchvision', 'timm>=1.0',
    'ultralytics>=8.2', 'onnx', 'onnxruntime',
    'imagehash', 'Pillow', 'numpy', 'scikit-learn', 'pandas',
    'opencv-python-headless', 'tqdm', 'matplotlib',
    'requests', 'datasets>=2.18',
])

import numpy as np, requests, torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms as T
from sklearn.metrics import f1_score, recall_score, confusion_matrix
from tqdm import tqdm

SEED = 0
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Sidecar JSON per crop — drives group-aware splits and error-analysis
# joins by source / capture_session without changing the model head.
@dataclass
class CropAnnotation:
    plant_id: str
    species: Optional[str]
    diagnosis: str
    severity: str               # low | medium | high | unknown
    leaf_or_whole_plant: str    # leaf | whole_plant
    capture_session: str
    source: str                 # inat | coco | plantvillage
    label_confidence: float     # in [0, 1]
    bbox: List[int]             # [x, y, w, h] — [0,0,0,0] when whole-image

def write_sidecar(p: Path, ann: CropAnnotation):
    p.with_suffix('.json').write_text(json.dumps(asdict(ann), ensure_ascii=False, indent=2))
def read_sidecar(p: Path):
    s = p.with_suffix('.json'); return json.loads(s.read_text()) if s.exists() else None

# Layout — splits are disjoint at plant_id level (group-aware splits in §2).
ROOT          = pathlib.Path('data_v2')
TRAIN_ROOT    = ROOT / 'train'
VAL_INDOMAIN  = ROOT / 'val' / 'indomain'
VAL_PRIMARY   = ROOT / 'val' / 'realworld'              # ship/no-ship gate
VAL_STUDIO    = ROOT / 'val' / 'plantvillage_studio'    # background-bias monitor
CONFORMAL_CAL = ROOT / 'val' / 'conformal_cal'

# v2.0: 7 trainable classes. The four physiological conditions
# (edema, leaf_scorch_underwatering, nutrient_deficiency,
# sunburn_or_scorch) are deferred to v2.1 — see intro.
CLASSES = [
    'bacterial_leaf_spot', 'gray_mold_botrytis', 'healthy', 'leaf_spot_fungal',
    'not_a_plant', 'powdery_mildew', 'viral_disease',
]
for c in CLASSES: (TRAIN_ROOT / c).mkdir(parents=True, exist_ok=True)
for d in (VAL_INDOMAIN, VAL_PRIMARY, VAL_STUDIO, CONFORMAL_CAL): d.mkdir(parents=True, exist_ok=True)

# Eval helpers — one evaluate(), three independent val loaders.
NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
EVAL_WORKERS = 2

def make_eval_loader(root, img_size, batch=128, workers=EVAL_WORKERS,
                     class_to_idx=None):
    """Build an eval loader whose label space matches training.

    ``ImageFolder`` indexes classes by sorted directory listing — so an
    eval slice missing any class (e.g. val_studio has no ``not_a_plant``
    because PlantVillage doesn't map there) silently shifts the labels
    of every higher-indexed class. ``class_to_idx`` is the training
    class map; we remap the targets so y_true is always comparable to
    model argmax across the 7 trained classes.
    """
    tx = T.Compose([T.Resize(int(img_size * 1.05)), T.CenterCrop(img_size), T.ToTensor(), NORM])
    target_transform = None
    if class_to_idx is not None:
        # Read the ImageFolder's class list once and translate per-sample
        # so PyTorch never sees a label outside [0, len(class_to_idx)).
        tmp = ImageFolder(str(root))
        slice_to_train = {
            tmp.class_to_idx[c]: class_to_idx[c]
            for c in tmp.classes if c in class_to_idx
        }
        if len(slice_to_train) != len(tmp.classes):
            unknown = [c for c in tmp.classes if c not in class_to_idx]
            raise RuntimeError(
                f'Eval folder {root} contains classes outside training '
                f'taxonomy: {unknown}. Refusing to silently mis-label.'
            )
        target_transform = slice_to_train.get
    ds = ImageFolder(str(root), transform=tx, target_transform=target_transform)
    if len(ds) == 0:
        raise RuntimeError(f'Eval folder is empty: {root}. Run §2 first.')
    return ds, DataLoader(ds, batch_size=batch, shuffle=False, num_workers=workers, pin_memory=True)

@torch.inference_mode()
def evaluate(model, dl, class_names, device, name=''):
    model.eval()
    y_true, y_pred = [], []
    for x, y in dl:
        y_pred.extend(model(x.to(device, non_blocking=True)).argmax(1).cpu().tolist())
        y_true.extend(y.tolist())
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    disease_idx = [i for i, n in enumerate(class_names) if n not in ('healthy', 'not_a_plant')]
    f1 = float(f1_score(y_true, y_pred, labels=disease_idx, average='macro', zero_division=0))
    recall = recall_score(y_true, y_pred, labels=list(range(len(class_names))), average=None, zero_division=0)
    h = class_names.index('healthy') if 'healthy' in class_names else None
    bias = float((y_pred == h).mean() / max((y_true == h).mean(), 1e-9)) if h is not None else 0.0
    print(f'[{name}] disease_macro_f1={f1:.3f}  healthy_bias={bias:.2f}  n={len(y_true)}')
    # Per-class recall is the noisy line — we only need it on the
    # primary ship/no-ship slice. Suppress on val_indomain / val_studio
    # to keep Colab cell output legible across long training runs.
    if 'val_realworld' in name:
        for c, r in zip(class_names, recall):
            print(f'    recall[{c:30s}] = {r:.3f}')
    return {'name': name, 'disease_macro_f1': f1, 'healthy_bias': bias,
            'per_class_recall': {c: float(r) for c, r in zip(class_names, recall)},
            'confusion_matrix': confusion_matrix(y_true, y_pred, labels=list(range(len(class_names)))).tolist()}


## §2 — Data acquisition (iNat + COCO + PlantVillage) and group splits

* **iNaturalist** — `quality_grade=research` + the documented numeric leaf annotation `term_id=36, term_value_id=38` ("Leaves: Green Leaves"). The previous `(17, 12)` pair was a bug: term 17 is "Alive or Dead" and 12 is the term-id of "Flowers and Fruits", so the API silently returned 0 results for every taxon and starved the healthy class. Verified against `/v1/controlled_terms`.
* **COCO val2017** — images without plant categories → `not_a_plant`.
* **PlantVillage** (HF mirror) — 38 lab labels mapped onto our taxonomy, used for `val_studio` only (background-bias monitor, never train).

The Money Plant Disease Atlas fetch was removed: the Mendeley DOI we previously referenced (`10.17632/2vc4mtps26.1`) does not resolve, and the dataset that does match the description (`kd8hs7ch6t`, "Money Plant Leaf datasets") is organised by `Healthy / Unhealthy / Yellow / Real / Combined / College premises` — none of which map onto our pathogen-level taxonomy. The old code path was a silent ~30 s no-op every run.

Photo downloads use a 4-retry exponential backoff and an 8-thread pool per taxon. Splits are group-aware (`plant_id`); `val_realworld` is carved first so it always wins ID conflicts.

In [ ]:
# §2 — Data acquisition + group-aware splits
UA = {'User-Agent': 'PlantVision/2.0 (research; github.com/pmixay/plantdiseases)'}
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
def iter_images(root: Path):
    for p in Path(root).rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS: yield p

def http_get_json(url, params=None, timeout=30, retries=3):
    last = None
    for _ in range(retries):
        try:
            r = requests.get(url, params=params, headers=UA, timeout=timeout)
            r.raise_for_status(); return r.json()
        except Exception as e:
            last = e; time.sleep(1.5)
    raise last

def http_download(url, dest, timeout=60):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0: return dest
    with requests.get(url, headers=UA, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        dest.parent.mkdir(parents=True, exist_ok=True)
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(8192):
                if chunk: f.write(chunk)
    return dest

def http_get_bytes(url, timeout=15, retries=4):
    """Fetch one image with retries — iNat CDN sporadically returns 429/503."""
    delay = 0.5
    for _ in range(retries):
        try:
            r = requests.get(url, headers=UA, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(delay); delay *= 2; continue
            r.raise_for_status(); return r.content
        except Exception:
            time.sleep(delay); delay *= 2
    return None

# ---- iNaturalist ---------------------------------------------------
INAT_BASE = 'https://api.inaturalist.org/v1'
# iNat 'Leaves' annotation: term_id=36, value_id=38 ('Green Leaves').
# The legacy (17, 12) pair was a bug — term 17 is 'Alive or Dead' and
# 12 is the term_id of 'Flowers and Fruits' (not a value at all).
# Verified against /v1/controlled_terms: with the bad pair the API
# returned **zero** observations for every taxon, which silently
# starved the healthy class out of the training set.
INAT_LEAF_TERM_ID, INAT_LEAF_TERM_VALUE_ID = 36, 38

def inat_resolve_taxon(name, rank='genus'):
    res = http_get_json(f'{INAT_BASE}/taxa', {'q': name, 'rank': rank}).get('results') or []
    return res[0]['id'] if res else None

def inat_observations(taxon_id, max_n, leaf_only=True):
    rows, page, per = [], 1, 100
    while len(rows) < max_n:
        params = {'taxon_id': taxon_id, 'quality_grade': 'research', 'photos': 'true',
                  'per_page': per, 'page': page, 'order': 'desc', 'order_by': 'created_at'}
        if leaf_only:
            params['term_id'] = INAT_LEAF_TERM_ID
            params['term_value_id'] = INAT_LEAF_TERM_VALUE_ID
        batch = (http_get_json(f'{INAT_BASE}/observations', params).get('results') or [])
        if not batch: break
        rows.extend(batch)
        if len(batch) < per: break
        page += 1; time.sleep(0.3)
    return rows[:max_n]

def _photo_url(photo, size='medium'):
    raw = photo.get('url') or ''
    return re.sub(r'/square\.', f'/{size}.', raw) if raw.startswith('http') else None

def _save_one(item, dest, klass, tx):
    obs, ph = item
    url = _photo_url(ph, 'medium')
    if not url: return 0
    fp = dest / f'inat_{tx}_{obs["id"]}_{ph["id"]}.jpg'
    if fp.exists(): return 0
    body = http_get_bytes(url)
    if not body: return 0
    fp.write_bytes(body)
    write_sidecar(fp, CropAnnotation(
        plant_id=str(obs.get('id')),
        species=(obs.get('taxon') or {}).get('name'),
        diagnosis=klass, severity='unknown', leaf_or_whole_plant='leaf',
        capture_session=f'inat_taxon_{tx}', source='inat',
        label_confidence=0.7, bbox=[0, 0, 0, 0]))
    return 1

def download_inat_class(taxon_ids, dest, klass, max_per_taxon=200, leaf_only=True, workers=8):
    dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for tx in taxon_ids:
        try: obs_list = inat_observations(tx, max_per_taxon, leaf_only=leaf_only)
        except Exception as e: print(f'  iNat taxon {tx} failed: {e}'); continue
        items = [(o, p) for o in obs_list for p in (o.get('photos') or [])[:1]]
        if not items: continue
        with ThreadPoolExecutor(max_workers=workers) as pool:
            for f in as_completed([pool.submit(_save_one, it, dest, klass, tx) for it in items]):
                n += f.result()
    return n

INAT_CLASS_TAXA = {
    'healthy': [('Ficus','genus'),('Monstera','genus'),('Epipremnum','genus'),('Calathea','genus'),
                ('Spathiphyllum','genus'),('Dracaena','genus'),('Sansevieria','genus'),('Philodendron','genus')],
    'powdery_mildew':      [('Erysiphales','order'),
                            ('Podosphaera','genus'),
                            ('Erysiphe','genus'),
                            ('Golovinomyces','genus')],
    'gray_mold_botrytis':  [('Botrytis cinerea','species'),('Botrytis','genus')],
    'leaf_spot_fungal':    [('Colletotrichum','genus'),('Cercospora','genus'),
                            ('Alternaria','genus'),('Septoria','genus')],
    'viral_disease':       [('Tobamovirus','genus'),('Geminiviridae','family'),('Begomovirus','genus')],
    'bacterial_leaf_spot': [('Xanthomonas','genus'),('Pseudomonas syringae','species')],
}

def fetch_inat_into_train(per_taxon=200):
    for klass, names in INAT_CLASS_TAXA.items():
        ids = [t for t in (inat_resolve_taxon(n, r) for n, r in names) if t]
        n = download_inat_class(ids, TRAIN_ROOT / klass, klass,
                                max_per_taxon=per_taxon, leaf_only=(klass == 'healthy'))
        print(f'iNat: {klass} += {n}')

# ---- COCO val2017 → not_a_plant -----------------------------------
COCO_CACHE = ROOT / 'cache_coco_val2017'
def fetch_coco_for_nap(target_n=1500):
    COCO_CACHE.mkdir(parents=True, exist_ok=True)
    img_zip, ann_zip = COCO_CACHE/'val2017.zip', COCO_CACHE/'annotations.zip'
    if not (COCO_CACHE/'val2017').exists():
        print('Downloading COCO val2017 (~1 GB)…')
        http_download('http://images.cocodataset.org/zips/val2017.zip', img_zip)
        with zipfile.ZipFile(img_zip) as z: z.extractall(COCO_CACHE)
    if not (COCO_CACHE/'annotations').exists():
        http_download('http://images.cocodataset.org/annotations/annotations_trainval2017.zip', ann_zip)
        with zipfile.ZipFile(ann_zip) as z: z.extractall(COCO_CACHE)
    ann = json.loads((COCO_CACHE/'annotations'/'instances_val2017.json').read_text())
    plant_cat = {c['id'] for c in ann['categories'] if c['name'] in {'potted plant','vase','broccoli','carrot'}}
    plant_imgs = {a['image_id'] for a in ann['annotations'] if a['category_id'] in plant_cat}
    eligible = [im for im in ann['images'] if im['id'] not in plant_imgs]
    rng = random.Random(42); rng.shuffle(eligible)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for im in eligible[:target_n]:
        src, dst = COCO_CACHE/'val2017'/im['file_name'], dest/f'coco_{im["id"]}.jpg'
        if not src.exists() or dst.exists(): continue
        shutil.copy(src, dst)
        write_sidecar(dst, CropAnnotation(plant_id=f'coco_{im["id"]}', species=None,
            diagnosis='not_a_plant', severity='low', leaf_or_whole_plant='whole_plant',
            capture_session='coco_val2017', source='coco', label_confidence=0.99, bbox=[0,0,0,0]))
        n += 1
    print(f'COCO: not_a_plant += {n}')

# ---- PlantVillage (HF) → val_studio ONLY -------------------------
# Roadmap-compliant: PV is a lab-condition dataset (Noyan 2022 — the
# leaf-on-uniform-background bias is a known studio confound). We
# keep the volume for the val_studio monitor only — every PV image
# lands in VAL_STUDIO, nothing leaks into train. Disease classes that
# need data come from iNat (see INAT_CLASS_TAXA above).
PV_LABEL_MAP = {
    'pepper__bell___bacterial_spot': 'bacterial_leaf_spot',
    'tomato___bacterial_spot':       'bacterial_leaf_spot',
    'peach___bacterial_spot':        'bacterial_leaf_spot',
    'apple___healthy': 'healthy', 'blueberry___healthy': 'healthy',
    'cherry_(including_sour)___healthy': 'healthy', 'corn_(maize)___healthy': 'healthy',
    'grape___healthy': 'healthy', 'pepper__bell___healthy': 'healthy',
    'potato___healthy': 'healthy', 'raspberry___healthy': 'healthy',
    'soybean___healthy': 'healthy', 'strawberry___healthy': 'healthy',
    'tomato___healthy': 'healthy',
    'corn_(maize)___cercospora_leaf_spot_gray_leaf_spot': 'leaf_spot_fungal',
    'tomato___septoria_leaf_spot': 'leaf_spot_fungal',
    'tomato___target_spot':        'leaf_spot_fungal',
    'tomato___leaf_mold':          'leaf_spot_fungal',
    'tomato___early_blight':       'leaf_spot_fungal',
    'tomato___late_blight':        'leaf_spot_fungal',
    'potato___early_blight':       'leaf_spot_fungal',
    'potato___late_blight':        'leaf_spot_fungal',
    'apple___apple_scab':          'leaf_spot_fungal',
    'apple___black_rot':           'leaf_spot_fungal',
    'apple___cedar_apple_rust':    'leaf_spot_fungal',
    'corn_(maize)___northern_leaf_blight': 'leaf_spot_fungal',
    'corn_(maize)___common_rust_': 'leaf_spot_fungal',
    'grape___black_rot':           'leaf_spot_fungal',
    'grape___esca_(black_measles)':'leaf_spot_fungal',
    'grape___leaf_blight_(isariopsis_leaf_spot)': 'leaf_spot_fungal',
    'orange___haunglongbing_(citrus_greening)':   'leaf_spot_fungal',
    'cherry_(including_sour)___powdery_mildew': 'powdery_mildew',
    'squash___powdery_mildew':                  'powdery_mildew',
    'tomato___tomato_mosaic_virus':           'viral_disease',
    'tomato___tomato_yellow_leaf_curl_virus': 'viral_disease',
    'tomato___spider_mites_two-spotted_spider_mite': None,  # pest, drop
}
def _pv_normalise(s): return re.sub(r'[^a-z0-9_]+', '_', s.strip().lower()).strip('_')

def fetch_plantvillage_into_studio(max_per_class=1500):
    """All PlantVillage images → VAL_STUDIO. Never train, never val_realworld.

    The function name was previously '..._into_train', a misnomer left
    over from when PV was a training source. v2.0 routes 100 % of PV
    to the studio-bias monitor, so the name now matches the behaviour.
    """
    try: from datasets import load_dataset
    except Exception: print('Install `datasets` to enable the PlantVillage fetch.'); return
    try:
        ds = load_dataset(
            'emre570/plantvillage-disease-classification',
            split='train', streaming=False,
            token=False,  # public dataset; ignore any stale HF_TOKEN in env
        )
    except Exception as e:
        print(f'PlantVillage HF download failed: {e}'); return
    label_names = ds.features['label'].names if 'label' in ds.features else None
    studio_counts, dropped = {}, 0
    rng = random.Random(0); order = list(range(len(ds))); rng.shuffle(order)
    for i in tqdm(order, desc='PlantVillage'):
        row = ds[i]
        raw = label_names[int(row['label'])] if label_names else (row.get('label_name') or str(row.get('label')))
        norm = _pv_normalise(raw)
        if norm not in PV_LABEL_MAP: dropped += 1; continue
        target = PV_LABEL_MAP[norm]
        if target is None or target not in CLASSES: dropped += 1; continue
        out_root = VAL_STUDIO / target
        out_root.mkdir(parents=True, exist_ok=True)
        if studio_counts.get(target, 0) >= max_per_class: continue
        stem = f'pv_{norm}_{i}'
        out = out_root / f'{stem}.jpg'
        if out.exists(): continue
        img = row['image']
        if hasattr(img, 'convert'): img.convert('RGB').save(out, quality=92)
        else: shutil.copy(img, out)
        write_sidecar(out, CropAnnotation(
            plant_id=stem, species=raw.split('___')[0].replace('_',' '),
            diagnosis=target, severity='unknown', leaf_or_whole_plant='leaf',
            capture_session='plantvillage', source='plantvillage',
            label_confidence=0.9, bbox=[0,0,0,0]))
        studio_counts[target] = studio_counts.get(target, 0) + 1
    print(f'PV val_studio: {studio_counts}\nPV dropped: {dropped}')

# Money Plant Disease Atlas fetch removed.
# Reason: the Mendeley DOI we previously referenced (10.17632/2vc4mtps26.1)
# does not exist (verified: 404 on the public-api endpoint) and the
# dataset that *does* match the description (kd8hs7ch6t, Money Plant
# Leaf datasets) is structured into folders {Healthy, Unhealthy,
# Yellow leaf, Real images, Combined, College premises} — none of
# which map onto our pathogen-level taxonomy. The previous code path
# silently fell through every run (download_url 404 → skip), so it
# was net zero training images for ~30s of wall clock. If a future
# release wants houseplant volume, prefer iNat queries on Epipremnum
# / Ficus / Monstera + the iNat 'Leaves: Green Leaves' annotation
# already wired through INAT_LEAF_TERM_ID/VALUE_ID above.

# ---- Group-aware splits -------------------------------------------
def make_group_split(dest_root, ratio, seed, label):
    """Move a group-aware fraction of train/<cls>/ into dest_root/<cls>/.

    Same plant_id (sidecar 'plant_id') always lands together — never
    split across train and the held-out slice."""
    rng = random.Random(seed); moved = 0
    for cls in CLASSES:
        cls_dir = TRAIN_ROOT / cls
        if not cls_dir.exists(): continue
        groups = {}
        for img in iter_images(cls_dir):
            pid = (read_sidecar(img) or {}).get('plant_id', img.stem)
            groups.setdefault(pid, []).append(img)
        ids = sorted(groups); rng.shuffle(ids)
        out = dest_root / cls; out.mkdir(parents=True, exist_ok=True)
        for pid in ids[:max(1, int(len(ids) * ratio))]:
            for img in groups[pid]:
                dst = out / img.name
                if dst.exists(): continue
                shutil.move(str(img), str(dst))
                sc = img.with_suffix('.json')
                if sc.exists(): shutil.move(str(sc), str(dst.with_suffix('.json')))
                moved += 1
    print(f'{label} populated  (moved={moved})')

# val_realworld first so it always wins plant_id conflicts.
def make_val_realworld(ratio=0.15, seed=2): make_group_split(VAL_PRIMARY,   ratio, seed, 'val_realworld')
def make_val_indomain(ratio=0.10, seed=0):  make_group_split(VAL_INDOMAIN,  ratio, seed, 'val_indomain')
# 10 %, not 5 %: with 7 classes and ~10k post-split, 5 % gave ~30–80
# cal samples per class — too thin for stable RAPS quantiles.
def make_conformal_cal(ratio=0.10, seed=1): make_group_split(CONFORMAL_CAL, ratio, seed, 'conformal_cal')

# ---- Run the pipeline ---------------------------------------------
fetch_inat_into_train(per_taxon=200)
fetch_coco_for_nap(target_n=1500)
fetch_plantvillage_into_studio(max_per_class=300)
make_val_realworld(); make_val_indomain(); make_conformal_cal()

def count_images(root): return sum(1 for _ in iter_images(root))
def report_class_counts(root):
    print(f'\n{root}:')
    for c in CLASSES:
        print(f'  {c:30s} {count_images(root/c) if (root/c).exists() else 0:5d}')
for r in (TRAIN_ROOT, VAL_PRIMARY, VAL_INDOMAIN): report_class_counts(r)

# Hard guard — empty class folders silently produce recall=0 in the
# final report and skew log-priors via clamp(min=1.0). Every class in
# the v2.0 taxonomy has a public source; emptiness is a real bug.
THRESH_TRAIN, THRESH_VAL = 30, 5
under_train = [c for c in CLASSES if count_images(TRAIN_ROOT/c) < THRESH_TRAIN]
under_val   = [c for c in CLASSES if count_images(VAL_PRIMARY/c) < THRESH_VAL]
if under_train or under_val:
    msg = []
    if under_train: msg.append(f'TRAIN_ROOT < {THRESH_TRAIN}: {under_train}')
    if under_val:   msg.append(f'VAL_PRIMARY < {THRESH_VAL}: {under_val}')
    raise RuntimeError('\n'.join(msg) + '\nExtend INAT_CLASS_TAXA and rerun.')


## §3 — Cleanup, Stage 1 detector, diagnostic probe

* **Dedup** SHA-256 → pHash (Hamming ≤ 8). Manual label-noise triage with Cleanlab / FiftyOne is intentionally kept off the runtime path so neither dep bloats the Colab session — install on demand if a cleanup pass is needed.
* **Stage 1** YOLOv8n is shipped pre-trained; this cell only re-trains when a `data.yaml` exists under `data_v2/`.
* **DINOv2-S frozen + linear probe** as a feature-quality sanity check. Probe ≫ student → bottleneck is data; probe ≈ student → bottleneck is capacity / augs.

In [ ]:
# §3 — Dedup + Stage 1 YOLO + DINOv2 probe
import imagehash
from PIL import Image as PILImage
from ultralytics import YOLO
import yaml, torch.nn as nn

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

def dedup_directory(root, phash_threshold=8):
    seen_sha, seen_phash, drop = set(), [], []
    for p in sorted(Path(root).rglob('*')):
        if not p.is_file() or p.suffix.lower() not in IMG_EXTS: continue
        s = sha256_file(p)
        if s in seen_sha: drop.append((p, 'sha256_dup')); continue
        seen_sha.add(s)
        try: ph = imagehash.phash(PILImage.open(p).convert('RGB'))
        except Exception: continue
        if any(ph - existing <= phash_threshold for existing in seen_phash):
            drop.append((p, 'phash_dup')); continue
        seen_phash.append(ph)
    for p, reason in drop:
        sc = p.with_suffix('.json')
        try: p.unlink(); sc.exists() and sc.unlink()
        except Exception: pass
    print(f'dedup({root}): dropped={len(drop)}')

# Dedup TRAIN_ROOT only — same-looking val_realworld photos from
# different plant_ids are a valid intra-class variability signal
# and shouldn't be pruned.
dedup_directory(TRAIN_ROOT)

# ---- Stage 1 — YOLOv8n single-class plant_leaf -------------------
def find_detector_yaml():
    if not ROOT.exists(): return None
    cands = list(ROOT.rglob('data.yaml'))
    return cands[0] if cands else None

yaml_path = find_detector_yaml()
if yaml_path is None:
    print(f'No YOLO dataset under {ROOT} — skipping Stage 1 training. '
          f'The shipped detector.{{pt,onnx}} will be used at inference.')
else:
    spec = yaml.safe_load(yaml_path.read_text())
    spec['names'], spec['nc'] = ['plant_leaf'], 1
    patched = yaml_path.parent / 'plant_leaf.yaml'
    patched.write_text(yaml.safe_dump(spec, sort_keys=False))
    YOLO('yolov8n.pt').train(
        data=str(patched), epochs=60, patience=12, batch=32, imgsz=640,
        optimizer='AdamW', lr0=1e-3, cos_lr=True, amp=True,
        mosaic=0.5, hsv_h=0.0, hsv_s=0.4, hsv_v=0.4,
        project='runs/detect', name='plant_leaf_indoor_v1',
    ).export(format='onnx', opset=13, dynamic=False)
    # Materialise trained weights into the bundle dir so the
    # operator only has to copy one tree into server/models/.
    Path('models').mkdir(exist_ok=True)
    for stem in ('best.pt', 'best.onnx'):
        src = next(Path('runs/detect').rglob(stem), None)
        if src: shutil.copy(src, f'models/detector.{stem.split(".")[-1]}')

# ---- DINOv2-S diagnostic probe -----------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
PROBE_SIZE = 224

def _ensure_populated(root, name):
    n = sum(1 for _ in iter_images(root))
    if n == 0: raise RuntimeError(f'{name} is empty — re-run §2.')
    return n
for r, n in [(TRAIN_ROOT, 'TRAIN_ROOT'), (VAL_PRIMARY, 'VAL_PRIMARY')]: _ensure_populated(r, n)

# The probe is a *training* loop on a frozen backbone — feed shuffled
# batches, otherwise the linear head sees class 0 .. class N in order
# and SGD never converges (the previous shuffle=False eval-style loader
# capped the probe at near-prior accuracy and made the diagnostic
# meaningless). The eval-time loader stays unshuffled.
PROBE_TRAIN_TX = T.Compose([
    T.Resize(int(PROBE_SIZE * 1.05)), T.CenterCrop(PROBE_SIZE),
    T.ToTensor(), NORM,
])
train_ds_probe = ImageFolder(str(TRAIN_ROOT), transform=PROBE_TRAIN_TX)
train_dl_probe = DataLoader(train_ds_probe, batch_size=256, shuffle=True,
                            num_workers=EVAL_WORKERS, pin_memory=True, drop_last=False)
CLASS_NAMES = train_ds_probe.classes
assert CLASS_NAMES == sorted(CLASSES), (CLASS_NAMES, sorted(CLASSES))
TRAIN_CLASS_TO_IDX = dict(train_ds_probe.class_to_idx)

val_ds_probe,   val_dl_probe   = make_eval_loader(
    VAL_PRIMARY, PROBE_SIZE, batch=256, class_to_idx=TRAIN_CLASS_TO_IDX,
)

backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').eval().to(device)
for p in backbone.parameters(): p.requires_grad_(False)
head = nn.Linear(384, len(CLASS_NAMES)).to(device)
opt = torch.optim.AdamW(head.parameters(), lr=3e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

for ep in range(8):
    head.train()
    for x, y in train_dl_probe:
        x, y = x.to(device, non_blocking=True), y.to(device)
        with torch.no_grad(): f = backbone(x)
        opt.zero_grad(); loss_fn(head(f), y).backward(); opt.step()
    head.eval(); correct = total = 0
    with torch.inference_mode():
        for x, y in val_dl_probe:
            correct += (head(backbone(x.to(device))).argmax(1) == y.to(device)).sum().item()
            total += y.numel()
    print(f'ep={ep+1}  val_realworld_top1={correct/total:.3f}')


## §4 — Loss, augs, teacher fine-tune

* **Logit Adjustment** (Menon ICLR 2021): `logits[y] -= τ · log π_y`, τ=1.0. Replaces the old inverse-frequency CE × `HEALTHY_DOWNWEIGHT=0.25` heuristic.
* **Indoor augs**: WB shifts, glare, defocus / motion blur, JPEG ringing. Disabled on purpose: strong hue shifts and `Solarize` — colour itself is a disease signal. MixUp / CutMix kept low.
* **Teacher**: `tf_efficientnetv2_s.in21k_ft_in1k`. Recipe: AdamW, weight_decay=1e-4, label_smoothing=0.05, cosine + 1-epoch warmup, 256 px, batch=128, **bf16 native** (no `GradScaler`). Head-only 3 ep (lr=3e-3) → unfreeze last 2 stages 12–15 ep (head 5e-4 / backbone 5e-5).

In [ ]:
# §4 — Loss + augs + teacher fine-tune
import timm, torch.nn.functional as F
from collections import Counter
from copy import deepcopy
from torch.amp import autocast
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from PIL import ImageFilter, ImageEnhance, Image as PILImage
from io import BytesIO

class LogitAdjustmentLoss(nn.Module):
    """Loss-side Logit Adjustment (Menon et al., ICLR 2021, eq. 4).

    During training: ``loss = CE(logits + tau * log pi, target)`` where
    ``pi`` is the empirical class prior. The trained network compensates
    in its weights, so at *inference* the raw logits are already
    calibrated for a uniform deployment distribution — no post-hoc
    subtraction is needed (and ``server/classifier.py`` correctly
    applies plain softmax to the logits).
    """
    def __init__(self, class_priors, tau=1.0, label_smoothing=0.05):
        super().__init__()
        self.register_buffer('log_prior', torch.log(class_priors.clamp(min=1e-12)))
        self.tau, self.ls = tau, label_smoothing
    def forward(self, logits, target):
        return F.cross_entropy(logits + self.tau * self.log_prior, target, label_smoothing=self.ls)

def class_priors_from_imagefolder(ds):
    counts = Counter([y for _, y in ds.samples])
    pri = torch.tensor([counts.get(i, 0) for i in range(len(ds.classes))], dtype=torch.float32)
    return pri.clamp(min=1.0) / pri.clamp(min=1.0).sum()

class WhiteBalanceShift:
    def __init__(self, p=0.5, strength=(0.8, 1.2)): self.p, self.lo, self.hi = p, *strength
    def __call__(self, img):
        if random.random() > self.p: return img
        r, g, b = img.split()
        r = r.point(lambda v: max(0, min(255, int(v * random.uniform(self.lo, self.hi)))))
        b = b.point(lambda v: max(0, min(255, int(v * random.uniform(self.lo, self.hi)))))
        return PILImage.merge('RGB', (r, g, b))

class Glare:
    def __init__(self, p=0.25): self.p = p
    def __call__(self, img):
        return img if random.random() > self.p else ImageEnhance.Brightness(img).enhance(random.uniform(1.05, 1.4))

class DefocusOrMotion:
    def __init__(self, p=0.3): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        f = ImageFilter.GaussianBlur(radius=random.uniform(0.8, 2.4)) if random.random() < 0.5 \
            else ImageFilter.BoxBlur(random.uniform(1.0, 2.5))
        return img.filter(f)

class JpegRecompress:
    def __init__(self, p=0.4, q_range=(45, 90)): self.p, self.q_range = p, q_range
    def __call__(self, img):
        if random.random() > self.p: return img
        buf = BytesIO(); img.save(buf, format='JPEG', quality=random.randint(*self.q_range))
        buf.seek(0); return PILImage.open(buf).convert('RGB')

indoor_train_tx = T.Compose([
    T.Resize(int(256 * 1.15)),
    T.RandomResizedCrop(256, scale=(0.6, 1.0), ratio=(0.85, 1.15)),
    T.RandomHorizontalFlip(),
    WhiteBalanceShift(p=0.5), Glare(p=0.25), DefocusOrMotion(p=0.3), JpegRecompress(p=0.4),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.0),
    T.ToTensor(), NORM, T.RandomErasing(p=0.15, scale=(0.02, 0.1)),
])

# ---- Teacher: EfficientNetV2-S ----------------------------------
BATCH, IMG = 128, 256
train_ds = ImageFolder(str(TRAIN_ROOT), transform=indoor_train_tx)
if len(train_ds) == 0: raise RuntimeError('TRAIN_ROOT empty — re-run §2.')
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=8,
                      pin_memory=True, drop_last=True)
# Train class map is the source of truth for label indices on every
# eval slice — see make_eval_loader docstring.
TRAIN_CLASS_TO_IDX = dict(train_ds.class_to_idx)
val_indomain_ds, val_indomain_dl   = make_eval_loader(
    VAL_INDOMAIN, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
val_realworld_ds, val_realworld_dl = make_eval_loader(
    VAL_PRIMARY,  IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
try:
    val_studio_ds, val_studio_dl = make_eval_loader(
        VAL_STUDIO, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
    )
except RuntimeError:
    val_studio_ds = val_studio_dl = None

priors = class_priors_from_imagefolder(train_ds).to(device)
loss_fn = LogitAdjustmentLoss(priors, tau=1.0)
teacher = timm.create_model('tf_efficientnetv2_s.in21k_ft_in1k',
                            pretrained=True, num_classes=len(train_ds.classes)).to(device)

def step_eval(model, tag):
    out = {'indomain': evaluate(model, val_indomain_dl, CLASS_NAMES, device, f'{tag} val_indomain'),
           'realworld': evaluate(model, val_realworld_dl, CLASS_NAMES, device, f'{tag} val_realworld')}
    if val_studio_dl is not None:
        out['studio'] = evaluate(model, val_studio_dl, CLASS_NAMES, device, f'{tag} val_studio')
    return out

def _make_sched(opt, n_epochs, n_steps_per_epoch):
    return SequentialLR(opt, schedulers=[
        LinearLR(opt, start_factor=0.1, total_iters=n_steps_per_epoch),
        CosineAnnealingLR(opt, T_max=max(1, (n_epochs - 1) * n_steps_per_epoch)),
    ], milestones=[n_steps_per_epoch])

def _train_phase(model, opt, n_epochs, scheduler, tag, early_stop=False, patience=4):
    best, plateau, best_state = -1.0, 0, None
    for ep in range(n_epochs):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device)
            with autocast(device_type='cuda', dtype=torch.bfloat16):
                loss = loss_fn(model(x), y)
            # bf16 + heavy augs (Glare × WB × JpegRecompress q=45)
            # rarely produces NaN/Inf; skip the batch instead of
            # silently corrupting the optimiser state.
            if not torch.isfinite(loss):
                print(f'WARN: non-finite loss at {tag} ep{ep+1}; skipping batch'); continue
            opt.zero_grad(); loss.backward(); opt.step(); scheduler.step()
        m = step_eval(model, f'{tag} ep{ep+1}')
        if early_stop:
            f1 = m['indomain']['disease_macro_f1']
            if f1 > best:
                best, plateau = f1, 0
                # Persist the best-seen weights so the phase returns the
                # peak indomain F1, not whatever the cosine tail produces.
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                plateau += 1
                if plateau >= patience:
                    print(f'early stop @ ep{ep+1}  (best f1={best:.3f})')
                    break
    if early_stop and best_state is not None:
        model.load_state_dict(best_state)
        print(f'restored best teacher weights  best_indomain_f1={best:.3f}')

# Phase 1 — head-only
for n, p in teacher.named_parameters(): p.requires_grad_('classifier' in n)
opt = torch.optim.AdamW([p for p in teacher.parameters() if p.requires_grad],
                        lr=3e-3, weight_decay=1e-4)
_train_phase(teacher, opt, 3, _make_sched(opt, 3, len(train_dl)), 'head')

# Phase 2 — unfreeze the last two backbone stages. EfficientNetV2-S
# in timm exposes blocks.0..blocks.5 (6 stages); the legacy 'blocks.6'
# selector silently expanded to nothing on this backbone, so we now
# discover the top stages dynamically and assert non-empty groups.
block_idxs = sorted({int(m.group(1)) for n in dict(teacher.named_parameters())
                     if (m := re.match(r'blocks\.(\d+)\.', n))})
if not block_idxs:
    raise RuntimeError('teacher has no `blocks.N.*` parameters — '
                       'unexpected backbone topology.')
top_blocks = [f'blocks.{i}' for i in block_idxs[-2:]]
unfreeze = top_blocks + ['classifier']
for p in teacher.parameters(): p.requires_grad_(False)
for n, p in teacher.named_parameters():
    if any(k in n for k in unfreeze): p.requires_grad_(True)
head_params     = [p for n, p in teacher.named_parameters() if 'classifier' in n and p.requires_grad]
backbone_params = [p for n, p in teacher.named_parameters()
                   if any(k in n for k in top_blocks) and p.requires_grad]
if not head_params or not backbone_params:
    raise RuntimeError(f'empty optimiser group: '
                       f'head={len(head_params)} backbone={len(backbone_params)}')
print(f'Phase 2 unfreeze: {top_blocks} + classifier  '
      f'(head={len(head_params)} params, backbone={len(backbone_params)} params)')
opt = torch.optim.AdamW([
    {'params': head_params,     'lr': 5e-4},
    {'params': backbone_params, 'lr': 5e-5},
], weight_decay=1e-4)
_train_phase(teacher, opt, 15, _make_sched(opt, 15, len(train_dl)), 'tune', early_stop=True)


## §5 — KD into MobileNetV3-Large student

KL-divergence on softened logits (T=4) + 0.3 × CE on hard labels. 8 epochs, head 5e-4 / backbone 5e-5. Gradient checkpointing on the student keeps the bf16 forward inside 40 GB even at 256 px.

In [ ]:
# §5 — Knowledge distillation
from torchvision import models

T_KD, ALPHA = 4.0, 0.7
student = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
student.classifier[3] = nn.Linear(student.classifier[3].in_features, len(train_ds.classes))
student = student.to(device)
teacher.eval()
opt = torch.optim.AdamW([
    {'params': student.classifier.parameters(), 'lr': 5e-4},
    {'params': [p for n, p in student.named_parameters() if 'classifier' not in n], 'lr': 5e-5},
], weight_decay=1e-4)
sched = _make_sched(opt, 8, len(train_dl))

# Track BEST val_indomain disease F1 — KD with τ=4 sometimes drifts
# in the last 1-2 epochs; we ship the best-seen weights, not the last.
from copy import deepcopy
best_f1, best_state, best_ep, plateau, PATIENCE = -1.0, None, 0, 0, 3
for ep in range(8):
    student.train()
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device)
        with autocast(device_type='cuda', dtype=torch.bfloat16):
            with torch.no_grad(): t_logits = teacher(x)
            s_logits = student(x)
            kd = F.kl_div(F.log_softmax(s_logits / T_KD, dim=1),
                          F.softmax(t_logits / T_KD, dim=1),
                          reduction='batchmean') * (T_KD * T_KD)
            ce = loss_fn(s_logits, y)
            loss = ALPHA * kd + (1 - ALPHA) * ce
        if not torch.isfinite(loss):
            print(f'WARN: non-finite KD loss at ep{ep+1}; skipping batch'); continue
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    metrics = step_eval(student, f'kd ep{ep+1}')
    f1 = metrics['indomain']['disease_macro_f1']
    if f1 > best_f1:
        best_f1, best_ep, plateau = f1, ep + 1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
    else:
        plateau += 1
        if plateau >= PATIENCE:
            print(f'KD early stop @ ep{ep+1}  (best ep={best_ep} f1={best_f1:.3f})')
            break

# Restore + persist the best checkpoint, never the last.
if best_state is not None:
    student.load_state_dict(best_state)
Path('models').mkdir(exist_ok=True)
torch.save(student.state_dict(), 'models/classifier.pth')
print(f'KD complete  best_indomain_f1={best_f1:.3f}  best_ep={best_ep}')


## §6 — ONNX, static INT8 (QDQ), temperature, conformal RAPS

* Export on **CPU** so the graph stays portable.
* Static INT8 calibration with 500 photos from `val_realworld`, then re-evaluate on the same slice. Drift > 0.05 aborts; drift > 0.02 only WARNs (static QDQ across MobileNetV3-Large Conv2d routinely costs 0.01–0.04 on a healthy run).
* **Quantisation type:** `weight_type=QInt8`, `activation_type=QUInt8` (U8S8). The previous `QInt8` activation choice silently fell back to FP32 on `CPUExecutionProvider` because ORT's quantised CPU kernels only support unsigned activations — quantisation produced an INT8 file but the runtime hot path stayed FP32, defeating the whole step.
* **Calibration is on the deployed INT8 ONNX session** (temperature + conformal RAPS + final reported metrics). Fitting `T` and `q_hat` on the FP32 GPU torch model and then shipping an INT8 CPU model used to drift coverage 3–6 pp below target.
* Temperature scaling on `val_realworld` NLL → `models/temperature.json`.
* Conformal RAPS on `conformal_cal` (10 % of train, group-disjoint) → `models/conformal.json`. Mirrors `server/conformal.py`.

In [ ]:
# §6 — ONNX FP32 → static INT8 + temperature + conformal RAPS
import onnxruntime as ort
from onnxruntime.quantization import (
    quantize_static, CalibrationDataReader, QuantFormat, QuantType)

# 1. ONNX FP32 export — on CPU to avoid baking CUDA-only ops in the graph.
student_cpu = student.cpu().eval()
torch.onnx.export(
    student_cpu, torch.randn(1, 3, IMG, IMG), 'models/classifier.onnx',
    opset_version=18, input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'N'}, 'logits': {0: 'N'}})

# 2. Static INT8 calibration — 500 real photos from val_realworld.
class RealWorldCalibrationReader(CalibrationDataReader):
    def __init__(self, dataset, n=500, input_name='input'):
        self.input_name = input_name
        idx = np.random.RandomState(0).permutation(len(dataset))[:n]
        self._iter = iter([(dataset[i][0].unsqueeze(0).numpy().astype(np.float32),) for i in idx])
    def get_next(self):
        nxt = next(self._iter, None)
        return None if nxt is None else {self.input_name: nxt[0]}

# ORT's CPUExecutionProvider only supports U8 *activations* for the
# quantised matmul/conv kernels — passing QInt8 here used to silently
# de-optimise the graph (kernels fall back to FP32 dequant→requant).
# Weights stay QInt8 with per-channel scales: this is the standard
# U8S8 recipe ORT recommends for x86 CPU.
quantize_static(
    'models/classifier.onnx', 'models/classifier_int8.onnx',
    calibration_data_reader=RealWorldCalibrationReader(val_realworld_ds, n=500),
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=True)

# 3. Mandatory post-quantization validation on val_realworld.
def _ort_session(path):
    return ort.InferenceSession(path, providers=['CPUExecutionProvider'])

def _ort_logits(sess, x_np):
    in_name = sess.get_inputs()[0].name
    return sess.run(None, {in_name: x_np})[0]

def eval_onnx(path, dl):
    sess = _ort_session(path)
    y_true, y_pred = [], []
    for x, y in dl:
        logits = _ort_logits(sess, x.numpy().astype(np.float32))
        y_pred.extend(logits.argmax(1).tolist())
        y_true.extend(y.tolist())
    disease_idx = [i for i, n in enumerate(CLASS_NAMES) if n not in ('healthy', 'not_a_plant')]
    return float(f1_score(y_true, y_pred, labels=disease_idx, average='macro', zero_division=0))

f1_fp32 = eval_onnx('models/classifier.onnx',      val_realworld_dl)
f1_int8 = eval_onnx('models/classifier_int8.onnx', val_realworld_dl)
drift = f1_fp32 - f1_int8
print(f'val_realworld disease F1  fp32={f1_fp32:.4f}  int8={f1_int8:.4f}  drift={drift:+.4f}')
WARN_DRIFT, FAIL_DRIFT = 0.02, 0.05
if drift > FAIL_DRIFT:
    raise RuntimeError(f'INT8 dropped F1 by {drift:.3f} (> {FAIL_DRIFT}). Fall back to FP32.')
elif drift > WARN_DRIFT:
    print(f'WARN: INT8 drift {drift:.3f} > {WARN_DRIFT}; shipping anyway.')

# All downstream calibration runs on the **deployed INT8 ONNX session**
# — fitting temperature on FP32 GPU logits and then shipping an INT8
# CPU model is the classic miscalibration trap (same input, ±0.5 logit
# per class once you swap backends, conformal coverage drops 3-6 pp).
int8_sess = _ort_session('models/classifier_int8.onnx')

def _collect_logits(sess, dl):
    all_logits, all_y = [], []
    for x, y in dl:
        all_logits.append(_ort_logits(sess, x.numpy().astype(np.float32)))
        all_y.append(y.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)

# 4. Temperature scaling on val_realworld NLL (INT8 logits).
#    Caveat: fitting T on the slice we also report disease F1 on is
#    a soft leakage by the letter of the eval contract. T does not
#    change argmax, so disease_macro_f1 stays honest; only NLL on
#    val_realworld is biased low. Fitting on val_indomain instead
#    would mismatch the production source distribution. v2.1 will
#    add a dedicated 5 %% T_CAL group split if this becomes blocking.
logits_val_np, y_val_np = _collect_logits(int8_sess, val_realworld_dl)
logits_val = torch.from_numpy(logits_val_np)
y_val      = torch.from_numpy(y_val_np).long()

T_param = torch.tensor(1.0, requires_grad=True)
opt = torch.optim.LBFGS([T_param], lr=0.05, max_iter=50)
def closure():
    opt.zero_grad()
    loss = F.cross_entropy(logits_val / T_param.clamp(min=0.05), y_val)
    loss.backward(); return loss
opt.step(closure)
T_val = float(T_param.detach())
Path('models/temperature.json').write_text(json.dumps({'temperature': T_val}, indent=2))
print(f'fitted T={T_val:.3f}  in-dist MaxLogit mean={logits_val.max(1).values.numpy().mean():.2f}')

# 5. Conformal RAPS — mirrors server/conformal.py.
def _raps_score(p_sorted, lam, k_reg):
    cs = np.cumsum(p_sorted)
    return cs + lam * np.maximum(0.0, np.arange(1, len(p_sorted)+1, dtype=np.float64) - k_reg)

def calibrate_q_hat(cal_probs, cal_labels, *, alpha=0.1, lambda_reg=0.01, k_reg=1):
    n = cal_probs.shape[0]
    nonconf = np.empty(n)
    for i in range(n):
        order = np.argsort(-cal_probs[i])
        s = _raps_score(cal_probs[i, order], lambda_reg, k_reg)
        nonconf[i] = s[int(np.where(order == cal_labels[i])[0][0])]
    q = float(np.quantile(nonconf, min(1.0, np.ceil((n+1) * (1-alpha)) / n), method='higher'))
    covered = 0
    for i in range(n):
        order = np.argsort(-cal_probs[i])
        s = _raps_score(cal_probs[i, order], lambda_reg, k_reg)
        above = np.where(s >= q)[0]
        size = max(int(above[0]) + 1 if above.size else len(s), 1)
        covered += int(cal_labels[i] in order[:size])
    return {'method': 'raps', 'alpha': alpha, 'q_hat': q,
            'lambda_reg': lambda_reg, 'k_reg': k_reg,
            'calibration_n': int(n), 'empirical_coverage': covered / n}

cal_ds, cal_dl = make_eval_loader(
    CONFORMAL_CAL, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
logits_cal_np, y_cal_np = _collect_logits(int8_sess, cal_dl)
probs_cal = torch.softmax(
    torch.from_numpy(logits_cal_np) / max(T_val, 1e-3), dim=1).numpy()
cfg = calibrate_q_hat(probs_cal, y_cal_np, alpha=0.10)
Path('models/conformal.json').write_text(json.dumps(cfg, indent=2))
print('conformal:', cfg)

# Move student back to GPU so anything below this point that still
# expects a GPU torch model continues to work.
student = student.to(device)

# 6. Persist the trained-on class list + img_size so the production
#    server (server/classifier.py) reads exactly what we trained.
#    Order matches sorted(ImageFolder.classes) by construction.
Path('models/classes.json').write_text(json.dumps({
    'class_names': CLASS_NAMES,
    'training': {'img_size': IMG},
}, indent=2))
print(f'classes.json: {len(CLASS_NAMES)} classes')

# 7. Final metrics on all three val slices via the **INT8 ONNX
#    session** — that's what production actually serves, so reporting
#    FP32 GPU torch numbers in model_card.json was a quiet form of
#    drift hiding (anything sub-WARN_DRIFT got swept under the rug).
class _OnnxModelAdapter:
    """Thin wrapper so evaluate() can call model(x) on an ORT session."""
    def __init__(self, sess): self.sess = sess
    def eval(self): return self
    def __call__(self, x):
        out = _ort_logits(self.sess, x.detach().cpu().numpy().astype(np.float32))
        return torch.from_numpy(out).to(x.device)

import datetime
int8_adapter = _OnnxModelAdapter(int8_sess)
final_metrics = {
    'indomain':  evaluate(int8_adapter, val_indomain_dl,  CLASS_NAMES, device, 'final val_indomain'),
    'realworld': evaluate(int8_adapter, val_realworld_dl, CLASS_NAMES, device, 'final val_realworld'),
}
if val_studio_dl is not None:
    final_metrics['studio'] = evaluate(
        int8_adapter, val_studio_dl, CLASS_NAMES, device, 'final val_studio')

Path('models/training_metrics.json').write_text(
    json.dumps(final_metrics, indent=2))

model_card = {
    'version': '2.0',
    'trained_at': datetime.datetime.now(datetime.timezone.utc)
                          .replace(microsecond=0).isoformat(),
    'classes': CLASS_NAMES,
    'img_size': IMG,
    'temperature': T_val,
    'conformal': cfg,
    'train_counts':         {c: count_images(TRAIN_ROOT  / c) for c in CLASS_NAMES},
    'val_realworld_counts': {c: count_images(VAL_PRIMARY / c) for c in CLASS_NAMES},
    'val_indomain_counts':  {c: count_images(VAL_INDOMAIN/ c) for c in CLASS_NAMES},
    'hyperparams': {
        'teacher': 'tf_efficientnetv2_s.in21k_ft_in1k',
        'student': 'mobilenet_v3_large',
        'kd_alpha': ALPHA, 'kd_tau': T_KD, 'la_tau': 1.0,
        'label_smoothing': 0.05, 'batch': BATCH, 'img_size': IMG,
    },
    'int8': {'drift': drift, 'f1_fp32': f1_fp32, 'f1_int8': f1_int8,
             'warn_drift': WARN_DRIFT, 'fail_drift': FAIL_DRIFT,
             'activation_type': 'QUInt8', 'weight_type': 'QInt8',
             'per_channel': True, 'format': 'QDQ'},
    'kd':   {'best_indomain_disease_f1': best_f1, 'best_epoch': best_ep},
    'reporting_backend': 'onnxruntime_int8_cpu',
    'reproducibility': {
        'seed': SEED,
        # cudnn.deterministic intentionally OFF — costs 10-15 %% wall
        # clock, returns ±0.01-0.03 F1 jitter between same-seed runs.
        # Flip to True before any v2.0-vs-v2.1 A/B comparison.
        'cudnn_deterministic': bool(getattr(
            torch.backends.cudnn, 'deterministic', False)),
        'cudnn_benchmark':     bool(getattr(
            torch.backends.cudnn, 'benchmark', True)),
    },
}
Path('models/model_card.json').write_text(json.dumps(model_card, indent=2))
print('Persisted: training_metrics.json + model_card.json')

# 8. Detector artefact contract — the bundle dir must contain
#    detector.{pt,onnx} or the server fails fast at startup. If §3
#    trained YOLO it already copied them; otherwise warn loudly so
#    the operator drops the shipped pair into models/ before zipping.
missing = [s for s in ('detector.pt', 'detector.onnx')
           if not Path(f'models/{s}').exists()]
if missing:
    print(f'WARN: {missing} not in models/ — drop the shipped '
          f'server/models/detector.{{pt,onnx}} pair into ./models/ '
          f'before assembling the deploy bundle, or re-run §3 with '
          f'a YOLO data.yaml under data_v2/.')


## §7 — Evaluation contract

Three independent slices, three reports per release:

| Slice | Role | Pass criterion |
|---|---|---|
| `val_indomain` | training-loop metric, drives early stopping | non-decreasing for ≥ 4 epochs |
| `val_realworld` | **primary — ship/no-ship gate** | disease macro-F1 ∈ **0.70–0.78**; no rare class with recall = 0; healthy-bias < 1.5 |
| `val_studio` | transfer / background-bias monitor (Noyan 2022) | tracked, never blocking |

Persisted artefacts (all written to `models/` in the notebook CWD):

* `classifier_int8.onnx`, `classifier.pth` — model weights.
* `classes.json` — `{class_names, training.img_size}`. Server reads it at startup so the trained-on taxonomy (7 in v2.0) is the contract; Android keeps its 11-entry disease lookup unchanged, only the 7 trained keys are ever queried.
* `temperature.json`, `conformal.json` — calibration configs.
* `detector.{pt,onnx}` — Stage 1 weights, copied from `runs/detect/.../weights/` if §3 retrained, otherwise the operator drops in the shipped pair (a clear WARN fires at the end of §6 if both are missing).
* `training_metrics.json` — structured per-class recall + confusion matrix on every val slice.
* `model_card.json` — provenance: train/val counts per class, hyperparams, INT8 drift, best-KD epoch, repro flags. Required for any post-deploy debugging without re-running the notebook.
